# 🗄️ Notebook 03 — Análise SQL Avançada: CPV, Giro e Tendências

**Projeto:** Análise de Custos e Margem por Categoria — Olist E-Commerce  
**Autor:** Ariel Marquezin  
**Stack:** DuckDB · pandas · numpy  

---

## 🎯 Objetivo

Aprofundar a análise financeira com SQL analítico avançado:

1. **Window Functions** — ROW_NUMBER, RANK, LAG, LEAD, SUM cumulativo
2. **Análise de Pareto** — quais categorias concentram 80% do CPV?
3. **Giro de estoque simulado** — lead time × custo de capital em trânsito
4. **Tendência MoM** — evolução mensal de receita e cancelamentos
5. **Matriz de risco** — combinação de margem × taxa de cancelamento

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

import duckdb
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

GOLD_PATH   = "../data/gold/"
PARQUET_PATH = "../data/parquet/"
os.makedirs(GOLD_PATH, exist_ok=True)

con = duckdb.connect()

# Registrar Parquet Bronze como views
for name in ["orders", "order_items", "products", "payments", "reviews", "customers"]:
    path = os.path.join(PARQUET_PATH, f"{name}.parquet")
    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_parquet('{path}')")

# Registrar Gold já calculado
gold_path = os.path.join(GOLD_PATH, "dre_por_categoria.parquet")
con.execute(f"CREATE OR REPLACE VIEW dre AS SELECT * FROM read_parquet('{gold_path}')")

print("DuckDB pronto ✅")

In [ ]:
# ── 2. Window Functions — Ranking triplo de performance ──────────────────────
# ROW_NUMBER + RANK + DENSE_RANK em receita, margem e EBITDA.

df_ranking = con.execute("""
SELECT
    category,
    receita_liquida,
    margem_bruta_media_pct,
    ebitda,
    cancel_rate_pct,

    ROW_NUMBER() OVER (ORDER BY receita_liquida       DESC) AS rn_receita,
    ROW_NUMBER() OVER (ORDER BY margem_bruta_media_pct DESC) AS rn_margem,
    ROW_NUMBER() OVER (ORDER BY ebitda                DESC) AS rn_ebitda,

    RANK()       OVER (ORDER BY cancel_rate_pct       DESC) AS rank_cancelamento,

    -- Score composto: média dos rankings (menor = melhor overall)
    ROUND((
        ROW_NUMBER() OVER (ORDER BY receita_liquida DESC)
      + ROW_NUMBER() OVER (ORDER BY margem_bruta_media_pct DESC)
      + ROW_NUMBER() OVER (ORDER BY ebitda DESC)
    ) / 3.0, 1)                                             AS score_composto

FROM dre
ORDER BY score_composto ASC
""").df()

print("Top 10 categorias por score composto (receita + margem + ebitda):")
print(df_ranking.head(10).to_string(index=False))

In [ ]:
# ── 3. Pareto do CPV — onde está o maior custo? ───────────────────────────────

df_pareto_cpv = con.execute("""
WITH base AS (
    SELECT *,
        cpv / SUM(cpv) OVER () * 100                   AS cpv_share_pct,
        SUM(cpv) OVER (
            ORDER BY cpv DESC
            ROWS UNBOUNDED PRECEDING
        ) / SUM(cpv) OVER () * 100                     AS cpv_acumulado_pct,
        ROW_NUMBER() OVER (ORDER BY cpv DESC)           AS rank
    FROM dre
)
SELECT
    rank,
    category,
    ROUND(cpv, 0)              AS cpv_total,
    ROUND(cpv_share_pct, 1)    AS cpv_share_pct,
    ROUND(cpv_acumulado_pct,1) AS cpv_acumulado_pct,
    ROUND(receita_liquida, 0)  AS receita_liquida,
    ROUND(margem_bruta_media_pct, 1) AS margem_pct,
    CASE
        WHEN cpv_acumulado_pct <= 80 THEN '🔴 Top 80% do CPV'
        ELSE '⚪ Restante'
    END                        AS pareto_grupo
FROM base
ORDER BY rank
""").df()

top80_cpv = df_pareto_cpv[df_pareto_cpv["pareto_grupo"] == "🔴 Top 80% do CPV"]
print(f"Pareto do CPV: {len(top80_cpv)} categorias concentram 80% do custo total")
print(df_pareto_cpv.head(15).to_string(index=False))

In [ ]:
# ── 4. Giro de Estoque Simulado + Custo de Capital em Trânsito ───────────────

df_giro = con.execute("""
WITH base AS (
    SELECT
        COALESCE(p.category, 'unknown')                  AS category,
        COUNT(*)                                         AS qtd_itens,
        ROUND(AVG(i.price), 2)                           AS ticket_medio,
        ROUND(SUM(i.price), 2)                           AS receita_total,
        ROUND(AVG(i.freight_value), 2)                   AS frete_medio,
        ROUND(AVG(
            DATE_DIFF('day',
                o.order_purchase_timestamp,
                o.order_delivered_customer_date)
        ), 1)                                            AS lead_time_dias
    FROM orders o
    JOIN order_items  i ON o.order_id   = i.order_id
    LEFT JOIN products p ON i.product_id = p.product_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
    GROUP BY category
    HAVING COUNT(*) >= 50
)
SELECT
    category,
    qtd_itens,
    ticket_medio,
    lead_time_dias,
    frete_medio,

    -- Giro simulado: itens vendidos / ticket médio (proxy de rotatividade)
    ROUND(qtd_itens / NULLIF(ticket_medio, 0), 2)        AS giro_simulado,

    -- Custo de capital em trânsito: R$2.50/dia × lead time
    ROUND(lead_time_dias * 2.50, 2)                      AS custo_transito_por_item,

    -- Custo total de trânsito na categoria
    ROUND(qtd_itens * lead_time_dias * 2.50, 0)          AS custo_transito_total,

    -- Ranking de eficiência logística
    RANK() OVER (ORDER BY lead_time_dias ASC)            AS rank_velocidade_entrega,
    RANK() OVER (ORDER BY giro_simulado  DESC)           AS rank_giro

FROM base
ORDER BY custo_transito_total DESC
""").df()

print("Top 10 categorias com maior custo de capital em trânsito:")
print(df_giro.head(10).to_string(index=False))

In [ ]:
# ── 5. Tendência Mensal com LAG e MoM ────────────────────────────────────────

df_monthly = con.execute("""
WITH monthly AS (
    SELECT
        DATE_TRUNC('month', o.order_purchase_timestamp)  AS mes,
        COUNT(DISTINCT o.order_id)                       AS total_pedidos,
        SUM(CASE WHEN o.order_status = 'canceled'
                 THEN 1 ELSE 0 END)                      AS cancelamentos,
        ROUND(SUM(i.price), 2)                           AS receita_bruta,
        ROUND(SUM(i.freight_value), 2)                   AS custo_frete_total,
        ROUND(AVG(i.price), 2)                           AS ticket_medio
    FROM orders o
    JOIN order_items i ON o.order_id = i.order_id
    GROUP BY mes
    HAVING COUNT(*) > 100
)
SELECT
    strftime(mes, '%Y-%m')                               AS mes,
    total_pedidos,
    cancelamentos,
    ROUND(cancelamentos * 100.0 / total_pedidos, 1)      AS cancel_rate_pct,
    receita_bruta,
    custo_frete_total,
    ticket_medio,

    -- MoM receita
    LAG(receita_bruta) OVER (ORDER BY mes)               AS receita_mes_anterior,
    ROUND(
        (receita_bruta - LAG(receita_bruta) OVER (ORDER BY mes))
        / NULLIF(LAG(receita_bruta) OVER (ORDER BY mes), 0) * 100,
    1)                                                   AS mom_receita_pct,

    -- MoM pedidos
    ROUND(
        (total_pedidos - LAG(total_pedidos) OVER (ORDER BY mes)) * 100.0
        / NULLIF(LAG(total_pedidos) OVER (ORDER BY mes), 0),
    1)                                                   AS mom_pedidos_pct,

    -- Acumulado de receita
    ROUND(SUM(receita_bruta) OVER (
        ORDER BY mes
        ROWS UNBOUNDED PRECEDING
    ), 2)                                                AS receita_acumulada

FROM monthly
ORDER BY mes
""").df()

print(f"Períodos analisados: {len(df_monthly)} meses")
df_monthly.tail(12)

In [ ]:
# ── 6. Matriz de Risco: Margem × Cancelamento ────────────────────────────────
# Identifica categorias com baixa margem E alta taxa de cancelamento.
# Essas são as mais críticas financeiramente.

df_risk = con.execute("""
SELECT
    category,
    margem_bruta_media_pct,
    cancel_rate_pct,
    ebitda,
    total_orders,
    ticket_medio,
    nps_medio,

    -- Score de risco financeiro (maior = mais preocupante)
    ROUND(
        (100 - margem_bruta_media_pct) * 0.5
      + cancel_rate_pct * 2.0
      + CASE WHEN ebitda < 0 THEN 20 ELSE 0 END,
    1)                                                   AS risk_score,

    CASE
        WHEN margem_bruta_media_pct < 15 AND cancel_rate_pct > 3
            THEN '🚨 Alto Risco'
        WHEN margem_bruta_media_pct < 20 OR cancel_rate_pct > 4
            THEN '⚠️  Atenção'
        WHEN margem_bruta_media_pct > 30 AND cancel_rate_pct < 2
            THEN '✅ Saudável'
        ELSE '🟡 Monitorar'
    END                                                  AS status_financeiro

FROM dre
ORDER BY risk_score DESC
""").df()

print("Distribuição por status financeiro:")
print(df_risk["status_financeiro"].value_counts().to_string())
print("\nTop 10 categorias de maior risco:")
print(df_risk.head(10)[["category","margem_bruta_media_pct","cancel_rate_pct","ebitda","status_financeiro"]].to_string(index=False))

In [ ]:
# ── 7. Exportar tabelas Gold adicionais ───────────────────────────────────────
import pyarrow as pa
import pyarrow.parquet as pq

def save_gold(df_out, name):
    path = os.path.join(GOLD_PATH, f"{name}.parquet")
    pq.write_table(pa.Table.from_pandas(df_out, preserve_index=False), path, compression="snappy")
    print(f"  Salvo: {name}.parquet ({len(df_out):,} linhas)")

save_gold(df_ranking,    "ranking_categorias")
save_gold(df_pareto_cpv, "pareto_cpv")
save_gold(df_giro,       "giro_estoque")
save_gold(df_monthly,    "tendencia_mensal")
save_gold(df_risk,       "matriz_risco")

print("\nNota book 03 concluído ✅ → próximo: 04_visualizacao.ipynb")

## ✅ Resumo — Notebook 03

| Análise | Técnica SQL | Resultado |
|---------|-------------|----------|
| Ranking triplo | ROW_NUMBER, RANK | Score composto por categoria |
| Pareto do CPV | SUM cumulativo OVER | X categorias = 80% do custo |
| Giro de estoque | AVG + RANK OVER | Lead time × custo de capital |
| Tendência MoM | LAG, SUM acumulado | Evolução mensal com variação % |
| Matriz de risco | CASE + score composto | Categorias críticas identificadas |

**Próximo →** `04_visualizacao.ipynb` — storytelling visual com Matplotlib + Seaborn.